# RescueVision - CPU pipeline smoke (synthetic, all 5 models)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
# pin albumentations: newer versions (preinstalled on the GPU image) have a
# stricter shape check that rejects our verified-consistent (HxWx3, HxW) pairs;
# --upgrade forces the downgrade past pip's "already satisfied"; all transforms
# ALSO use is_check_shapes=False as a version-proof second layer
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=False)
import segmentation_models_pytorch as smp
print("smp", smp.__version__)
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_workers=0: DataLoader workers died on the Kaggle GPU image; pin_memory
# off too (useless with workers=0, and a hard-kill vector on some drivers)
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

# per-epoch heartbeat: every trainer that logs an epoch also lands in the marks,
# so a silent kill still tells us how far training got and with what metrics
import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
# ---- synthetic mini processed_xbd tree (CPU smoke only; real runs get the tar) ----
import numpy as np, pandas as pd
from PIL import Image

rng = np.random.default_rng(0)
PROC = Path("/tmp/processed_xbd")
for sub in ["images/pre", "images/post", "masks/building_binary", "masks/damage_multiclass",
            "masks/change_binary", "crops/building_pre", "crops/building_post", "metadata"]:
    (PROC / sub).mkdir(parents=True, exist_ok=True)

events = [("guatemala-volcano", "volcano"), ("midwest-flooding", "flooding"), ("santa-rosa-wildfire", "fire")]
split_of_event = {"guatemala-volcano": "train", "midwest-flooding": "val", "santa-rosa-wildfire": "test"}
split_rows, crop_rows = [], []
crop_i = 0
label_names4 = ["no-damage", "minor-damage", "major-damage", "destroyed"]
for ev, ev_type in events:
    for t in range(8):
        tid = f"{ev}_{t:04d}"
        Image.fromarray(rng.integers(0, 255, (160, 160, 3), dtype=np.uint8)).save(PROC / "images/pre" / f"{tid}.png")
        Image.fromarray(rng.integers(0, 255, (160, 160, 3), dtype=np.uint8)).save(PROC / "images/post" / f"{tid}.png")
        Image.fromarray((rng.random((160, 160)) > 0.7).astype(np.uint8)).save(PROC / "masks/building_binary" / f"{tid}.png")
        dmg = rng.choice([0, 1, 2, 3, 4, 255], p=[.3, .25, .2, .1, .1, .05], size=(160, 160)).astype(np.uint8)
        Image.fromarray(dmg).save(PROC / "masks/damage_multiclass" / f"{tid}.png")
        Image.fromarray((rng.random((160, 160)) > 0.75).astype(np.uint8)).save(PROC / "masks/change_binary" / f"{tid}.png")
        split_rows.append({"tile_id": tid, "split": split_of_event[ev], "disaster_name": ev})
        for c in range(2):   # 2 crops per tile, balanced labels
            cid = f"crop_{crop_i:07d}"; crop_i += 1
            lab = 1 + ((t + c) % 4)
            Image.fromarray(rng.integers(0, 255, (64, 64, 3), dtype=np.uint8)).save(PROC / "crops/building_pre" / f"{cid}.png")
            Image.fromarray(rng.integers(0, 255, (64, 64, 3), dtype=np.uint8)).save(PROC / "crops/building_post" / f"{cid}.png")
            crop_rows.append({"crop_id": cid, "tile_id": tid, "label": label_names4[lab - 1], "label_id": lab,
                              "area": 100, "disaster_name": ev, "disaster_type": ev_type,
                              "pre_crop_path": f"crops/building_pre/{cid}.png",
                              "post_crop_path": f"crops/building_post/{cid}.png"})
pd.DataFrame(split_rows).to_csv(PROC / "metadata/splits_event_aware.csv", index=False)
pd.DataFrame(crop_rows).to_csv(PROC / "metadata/buildings.csv", index=False)
print("synthetic xBD tree:", len(split_rows), "tiles,", len(crop_rows), "crops")
NW = 2
mark("synthetic:xbd-done")

In [ ]:
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD
from rescuevision.data import datasets as _D
from PIL import Image as _PIL
from pathlib import Path as _PPath

def _load_arr(path):
    """Read .png or .npy array transparently."""
    path = _PPath(path)
    if path.suffix == ".npy":
        return np.load(path)
    return np.array(_PIL.open(path))

def _find_tile(directory, tile_id):
    for ext in (".png", ".npy"):
        p = _PPath(directory) / f"{tile_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{tile_id} in {directory}")

def joint_seg_transforms(cfg, phase="train", image_size=512):
    """Joint pipeline for a stacked pre+post pair (H, W, 6): one geometric draw for both."""
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(image_size, image_size)]
    if phase == "train":
        if cfg.get("horizontal_flip", 0) > 0:
            tf.append(A.HorizontalFlip(p=cfg["horizontal_flip"]))
        if cfg.get("vertical_flip", 0) > 0:
            tf.append(A.VerticalFlip(p=cfg["vertical_flip"]))
        if cfg.get("rotate90", 0) > 0:
            tf.append(A.RandomRotate90(p=cfg["rotate90"]))
        if cfg.get("brightness_contrast", 0) > 0:
            tf.append(A.RandomBrightnessContrast(p=cfg["brightness_contrast"]))
    if cfg.get("normalize", True):
        tf.append(A.Normalize(mean=mean, std=std))
    tf.append(ToTensorV2())
    return A.Compose(tf, is_check_shapes=False)

def joint_clf_pair_transforms(cfg, phase="train", image_size=224):
    """Joint pipeline for stacked pre+post crops (H, W, 6)."""
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(image_size, image_size)]
    if phase == "train":
        if cfg.get("horizontal_flip", 0) > 0:
            tf.append(A.HorizontalFlip(p=cfg["horizontal_flip"]))
        if cfg.get("vertical_flip", 0) > 0:
            tf.append(A.VerticalFlip(p=cfg["vertical_flip"]))
        if cfg.get("rotate", 0) > 0:
            tf.append(A.Rotate(limit=cfg["rotate"], p=0.5))
        if cfg.get("brightness_contrast", 0) > 0:
            tf.append(A.RandomBrightnessContrast(p=cfg["brightness_contrast"]))
    if cfg.get("normalize", True):
        tf.append(A.Normalize(mean=mean, std=std))
    tf.append(ToTensorV2())
    return A.Compose(tf, is_check_shapes=False)

def _nocheck(tf):
    # disable albumentations' shape pre-check: newer builds (preinstalled on the
    # Kaggle GPU image) reject our verified HxW-consistent image/mask pairs
    try:
        return A.Compose(tf.transforms, is_check_shapes=False)
    except Exception:
        return tf

def _joint_dmg_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    pre_image = _load_arr(_find_tile(self.pre_dir, tile_id))
    post_image = _load_arr(_find_tile(self.post_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    pair = np.concatenate([pre_image, post_image], axis=2)
    if self.transform:
        augmented = self.transform(image=pair, mask=mask)
        image, mask = augmented["image"], augmented["mask"]
    else:
        image = torch.from_numpy(pair.transpose(2, 0, 1))
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

def _joint_change_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    pre_image = _load_arr(_find_tile(self.pre_dir, tile_id))
    post_image = _load_arr(_find_tile(self.post_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    pair = np.concatenate([pre_image, post_image], axis=2)
    if self.transform:
        augmented = self.transform(image=pair, mask=mask)
        pair_t, mask = augmented["image"], augmented["mask"]
    else:
        pair_t = torch.from_numpy(pair.transpose(2, 0, 1))
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"pre_image": pair_t[:3], "post_image": pair_t[3:], "mask": mask, "tile_id": tile_id}

_D.XBDDamageSegmentationDataset.__getitem__ = _joint_dmg_seg_getitem
_D.XBDChangeDetectionDataset.__getitem__ = _joint_change_getitem

def _tile_dataset_init(self, processed_dir, split_csv, split="train", mask_type="building_binary",
                       transform=None, image_size=512, debug_limit=None):
    import pandas as pd
    self.processed_dir = _PPath(processed_dir)
    self.images_dir = self.processed_dir / "images" / "pre"
    self.pre_dir = self.processed_dir / "images" / "pre"
    self.post_dir = self.processed_dir / "images" / "post"
    if type(self).__name__ == "XBDDamageSegmentationDataset":
        self.masks_dir = self.processed_dir / "masks" / "damage_multiclass"
    elif type(self).__name__ == "XBDChangeDetectionDataset":
        self.masks_dir = self.processed_dir / "masks" / "change_binary"
    else:
        self.masks_dir = self.processed_dir / "masks" / mask_type
    self.transform = transform
    self.image_size = image_size
    splits_df = pd.read_csv(split_csv)
    tile_ids = splits_df[splits_df["split"] == split]["tile_id"].tolist()
    if debug_limit:
        tile_ids = tile_ids[:debug_limit]
    self.tile_ids = [t for t in tile_ids
                     if _find_tile(self.pre_dir, t).exists()
                     and _find_tile(self.post_dir, t).exists()
                     and _find_tile(self.masks_dir, t).exists()]
    print(f"{type(self).__name__}: split={split}, {len(self.tile_ids)} tiles")

def _xbd_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    image = _load_arr(_find_tile(self.pre_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    if self.transform:
        aug = self.transform(image=image, mask=mask)
        image, mask = aug["image"], aug["mask"]
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

_D.XBDSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDDamageSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDChangeDetectionDataset.__init__ = _tile_dataset_init
_D.XBDSegmentationDataset.__getitem__ = _xbd_seg_getitem
mark("patch:done")
print("png-aware dataset init patch applied")
print("joint pre/post transform patch applied")

In [ ]:
# ==== building seg (smoke) ====

In [ ]:
import time
import pandas as pd
mark("tar:wait")
TAR = None
if (Path("/tmp/processed_xbd") / "metadata" / "splits_event_aware.csv").exists():
    print("processed tree already at /tmp/processed_xbd (smoke/debug path)")
else:
    for _attempt in range(40):   # up to 20 min for the kernel-source mount
        for _p in Path("/kaggle/input").rglob("processed_xbd.tar"):
            TAR = _p
            break
        if TAR is not None:
            break
        _ents = [p.name for p in Path("/kaggle/input").iterdir()] if Path("/kaggle/input").exists() else []
        print(f"[{_attempt+1}/40] waiting for tar mount, /kaggle/input: {_ents}", flush=True)
        time.sleep(30)
    assert TAR is not None, "processed_xbd.tar never mounted - kernel-source attach failed"
    _gib = TAR.stat().st_size / 1e9
    print("tar:", TAR, f"{_gib:.2f} GB")
    mark("tar:mounted", extra=f"{_gib:.2f}GB")
    subprocess.run(["tar", "-xf", str(TAR), "-C", "/tmp"], check=True)
PROCESSED = Path("/tmp/processed_xbd")
assert (PROCESSED / "metadata" / "splits_event_aware.csv").exists(), "extracted tree missing splits csv"
_splits = pd.read_csv(PROCESSED / "metadata" / "splits_event_aware.csv")
print("tiles per split:", _splits["split"].value_counts().to_dict())
mark("tar:extracted", extra=str(_splits["split"].value_counts().to_dict()))

In [ ]:
import logging
import pandas as pd
from torch.utils.data import DataLoader
from rescuevision.data.datasets import XBDSegmentationDataset
from rescuevision.data.transforms import get_segmentation_transforms
from rescuevision.models.unet import create_unet
from rescuevision.models.losses import BCEDiceLoss
from rescuevision.training.train_segmentation import train_segmentation, evaluate_segmentation
from rescuevision.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
OUTPUT = WORK
name = "building_seg_unet_rn34"
METRIC = "iou"

SEED = 42
AUG = {"horizontal_flip": 0.5, "vertical_flip": 0.5, "rotate90": 0.5, "brightness_contrast": 0.3, "normalize": True}
BS = 4
EPOCHS = 1
PATIENCE = 1

config = {
    "seed": SEED, "device": DEVICE.type,
    "model": {"encoder": "resnet34", "encoder_weights": "imagenet"},
    "training": {"epochs": EPOCHS, "lr": 1e-4, "weight_decay": 1e-5, "scheduler": "cosine",
                  "mixed_precision": True, "early_stopping_patience": PATIENCE,
                  "save_best_metric": METRIC},
}

set_seed(SEED)
split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"
train_ds = XBDSegmentationDataset(PROCESSED, split_csv, split="train",
                                  mask_type="building_binary",
                                  transform=_nocheck(get_segmentation_transforms(AUG, "train", 512)))
val_ds = XBDSegmentationDataset(PROCESSED, split_csv, split="val",
                                mask_type="building_binary",
                                transform=_nocheck(get_segmentation_transforms(AUG, "val", 512)))
test_ds = XBDSegmentationDataset(PROCESSED, split_csv, split="test",
                                 mask_type="building_binary",
                                 transform=_nocheck(get_segmentation_transforms(AUG, "val", 512)))
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
mark("building:data", extra=f"train={len(train_ds)}")

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
test_loader = DataLoader(test_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False)

model = create_unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)
loss_fn = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
mark("building:model-ready")


# first-batch GPU probe: an OOM or crash here lands in the marks WITH a traceback
try:
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        _img = _b["image"].to(DEVICE)
        _m = _b["mask"].to(DEVICE)
        with torch.amp.autocast(DEVICE.type, enabled=True):
            _lg = model(_img)
            _loss = loss_fn(_lg.squeeze(1) if _lg.shape[1] == 1 else _lg,
                            _m.float() if _lg.shape[1] == 1 else _m)
        _loss.backward()
        model.zero_grad(set_to_none=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        mark("probe:ok", extra=f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    else:
        mark("probe:ok", extra="cpu")
except Exception:
    mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
    raise
mark("building:train-start", extra=f"train={len(train_ds)}")
best = train_segmentation(model, train_loader, val_loader, loss_fn, config,
                          output_dir=str(Path("/tmp/run") / name), is_binary=True, num_classes=1)
print("best:", best)
mark("building:train-done", extra=str(best))

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/building_seg_unet_rn34", WORK / "building_seg_unet_rn34", dirs_exist_ok=True)
print("results copied to", WORK / "building_seg_unet_rn34")
mark("copyout:building_seg_unet_rn34")
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log_df = pd.read_csv(next((WORK / "building_seg_unet_rn34" / "logs").glob("*.csv")))
metric = "iou"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
if "val_loss" in log_df:
    axes[0].plot(log_df["epoch"], log_df["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(log_df["epoch"], log_df[metric], color="tab:green")
axes[1].set_title(f"val {metric} (best={log_df[metric].max():.4f} @ epoch {int(log_df.loc[log_df[metric].idxmax(), 'epoch'])})")
axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK / "building_seg_unet_rn34" / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import json

ckpt = torch.load(WORK / name / "checkpoints" / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
print(f"best checkpoint: epoch {ckpt['epoch']}, {METRIC}={ckpt.get(f'best_{METRIC}'):.4f}")

summary = {
    "run": name,
    "best_epoch": int(ckpt["epoch"]),
    f"best_val_{METRIC}": float(ckpt.get(f"best_{METRIC}")),
    "history_best": float(pd.read_csv(next((WORK / name / "logs").glob("*.csv")))[METRIC].max()),
}
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

val_metrics = evaluate_segmentation(model, val_loader, DEVICE, is_binary=True, num_classes=1)
test_metrics = evaluate_segmentation(model, test_loader, DEVICE, is_binary=True, num_classes=1)
print("VAL :", {k: round(v, 4) for k, v in val_metrics.items()})
print("TEST:", {k: round(v, 4) for k, v in test_metrics.items()})
summary.update({"val": {k: round(v, 4) for k, v in val_metrics.items()},
                "test": {k: round(v, 4) for k, v in test_metrics.items()}})
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

import numpy as np, matplotlib.pyplot as plt
batch = next(iter(test_loader))
with torch.no_grad():
    pred = (torch.sigmoid(model(batch["image"].to(DEVICE))) > 0.5).cpu().numpy()
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i in range(min(4, batch["image"].shape[0])):
    img = batch["image"][i].numpy().transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    axes[0, i].imshow(img); axes[0, i].set_title(batch["tile_id"][i][:28])
    axes[1, i].imshow(batch["mask"][i], cmap="gray"); axes[1, i].set_title("GT")
    axes[2, i].imshow(pred[i, 0], cmap="gray"); axes[2, i].set_title("pred")
    for ax in axes[:, i]: ax.axis("off")
plt.tight_layout(); plt.savefig(WORK / name / "test_predictions.png", dpi=120, bbox_inches="tight")
plt.show()
mark("building:eval-done")

In [ ]:
# ==== damage seg (smoke) ====

In [ ]:
import logging
import pandas as pd
from torch.utils.data import DataLoader
from rescuevision.data.datasets import XBDDamageSegmentationDataset
from rescuevision.models.unet import create_unet
from rescuevision.models.losses import WeightedCEDiceLoss
from rescuevision.training.train_segmentation import train_segmentation, evaluate_segmentation
from rescuevision.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
OUTPUT = WORK
name = "damage_seg_unet_effb3_6ch"
METRIC = "miou"

SEED = 42
AUG = {"horizontal_flip": 0.5, "vertical_flip": 0.5, "rotate90": 0.5, "brightness_contrast": 0.3, "normalize": True}
BS = 2
EPOCHS = 1
PATIENCE = 1

config = {
    "seed": SEED, "device": DEVICE.type,
    "training": {"epochs": EPOCHS, "lr": 5e-5, "weight_decay": 1e-5, "scheduler": "cosine",
                  "mixed_precision": True, "early_stopping_patience": PATIENCE,
                  "save_best_metric": METRIC, "gradient_accumulation_steps": 1},
}

set_seed(SEED)
split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"
train_ds = XBDDamageSegmentationDataset(PROCESSED, split_csv, split="train",
                                        transform=joint_seg_transforms(AUG, "train", 512))
val_ds = XBDDamageSegmentationDataset(PROCESSED, split_csv, split="val",
                                      transform=joint_seg_transforms(AUG, "val", 512))
test_ds = XBDDamageSegmentationDataset(PROCESSED, split_csv, split="test",
                                       transform=joint_seg_transforms(AUG, "val", 512))
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
mark("damage:data", extra=f"train={len(train_ds)}")

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
test_loader = DataLoader(test_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False)

# class weights from PNG-or-NPY masks via the transparent loader (PNG preprocessing!)
counts = np.zeros(5, dtype=np.int64)
for tid in train_ds.tile_ids[:400]:
    m = _load_arr(_find_tile(PROCESSED / "masks/damage_multiclass", tid))
    counts += np.bincount(np.asarray(m, dtype=np.int64)[m != 255].flatten(), minlength=5)
print("pixel class counts:", counts.tolist())
assert counts.sum() > 0, "no mask pixels counted - mask path bug"
# recompute class counts from the extracted tiles (self-contained cell)
counts = np.zeros(NUM_FLOODNET_CLASSES, dtype=np.int64)
for _f in (TILES / "train").glob("*_ids.npy"):
    counts += np.bincount(np.load(_f).flatten(), minlength=NUM_FLOODNET_CLASSES)
print("class pixel counts:", counts.tolist())
assert counts.sum() > 0
freq = counts / counts.sum()
weights = 1.0 / np.sqrt(freq)          # softened inverse frequency
weights = weights / weights.sum() * 5
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
print("class weights:", class_weights.tolist())

model = create_unet(encoder_name="efficientnet-b3", encoder_weights=None, in_channels=6, classes=5).to(DEVICE)

# adapt pretrained 3-ch conv_stem to 6 channels (first 3 keep weights, rest get the mean)
def _stem(enc):
    # SMP renamed conv_stem -> _conv_stem in newer releases
    return enc.conv_stem if hasattr(enc, "conv_stem") else enc._conv_stem

_pre_enc = smp.Unet(encoder_name="efficientnet-b3", encoder_weights="imagenet",
                    in_channels=3, classes=5).encoder
pretrained = _stem(_pre_enc).weight.data
conv_stem = _stem(model.encoder)
new_w = conv_stem.weight.data.clone()
new_w[:, :3] = pretrained
new_w[:, 3:] = pretrained.mean(dim=1, keepdim=True).expand_as(new_w[:, 3:])
conv_stem.weight.data = new_w
del _pre_enc
print("conv_stem adapted 3 -> 6 channels")
mark("damage:model-ready")

loss_fn = WeightedCEDiceLoss(class_weights=class_weights, ce_weight=0.5, dice_weight=0.5, ignore_index=255)


# first-batch GPU probe: an OOM or crash here lands in the marks WITH a traceback
try:
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        _img = _b["image"].to(DEVICE)
        _m = _b["mask"].to(DEVICE)
        with torch.amp.autocast(DEVICE.type, enabled=True):
            _lg = model(_img)
            _loss = loss_fn(_lg.squeeze(1) if _lg.shape[1] == 1 else _lg,
                            _m.float() if _lg.shape[1] == 1 else _m)
        _loss.backward()
        model.zero_grad(set_to_none=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        mark("probe:ok", extra=f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    else:
        mark("probe:ok", extra="cpu")
except Exception:
    mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
    raise
mark("damage:train-start", extra=f"train={len(train_ds)}")
best = train_segmentation(model, train_loader, val_loader, loss_fn, config,
                          output_dir=str(Path("/tmp/run") / name), is_binary=False, num_classes=5, ignore_index=255)
print("best:", best)
mark("damage:train-done", extra=str(best))

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/damage_seg_unet_effb3_6ch", WORK / "damage_seg_unet_effb3_6ch", dirs_exist_ok=True)
print("results copied to", WORK / "damage_seg_unet_effb3_6ch")
mark("copyout:damage_seg_unet_effb3_6ch")
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log_df = pd.read_csv(next((WORK / "damage_seg_unet_effb3_6ch" / "logs").glob("*.csv")))
metric = "miou"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
if "val_loss" in log_df:
    axes[0].plot(log_df["epoch"], log_df["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(log_df["epoch"], log_df[metric], color="tab:green")
axes[1].set_title(f"val {metric} (best={log_df[metric].max():.4f} @ epoch {int(log_df.loc[log_df[metric].idxmax(), 'epoch'])})")
axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK / "damage_seg_unet_effb3_6ch" / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import json

ckpt = torch.load(WORK / name / "checkpoints" / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
print(f"best checkpoint: epoch {ckpt['epoch']}, {METRIC}={ckpt.get(f'best_{METRIC}'):.4f}")

summary = {
    "run": name,
    "best_epoch": int(ckpt["epoch"]),
    f"best_val_{METRIC}": float(ckpt.get(f"best_{METRIC}")),
    "history_best": float(pd.read_csv(next((WORK / name / "logs").glob("*.csv")))[METRIC].max()),
}
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

import numpy as np
val_metrics = evaluate_segmentation(model, val_loader, DEVICE, is_binary=False, num_classes=5, ignore_index=255)
test_metrics = evaluate_segmentation(model, test_loader, DEVICE, is_binary=False, num_classes=5, ignore_index=255)
print("VAL :", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in val_metrics.items()})
print("TEST:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in test_metrics.items()})
summary.update({"val": {k: round(v, 4) for k, v in val_metrics.items() if not isinstance(v, list)},
                "test": {k: round(v, 4) for k, v in test_metrics.items() if not isinstance(v, list)}})
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

from rescuevision.evaluation.confusion import compute_confusion_matrix, plot_confusion_matrix
preds, targets = [], []
with torch.no_grad():
    for batch in test_loader:
        p = model(batch["image"].to(DEVICE)).argmax(1).cpu().numpy()
        preds.append(p.flatten()); targets.append(batch["mask"].numpy().flatten())
cm = compute_confusion_matrix(np.concatenate(preds), np.concatenate(targets), num_classes=5, ignore_index=255)
plot_confusion_matrix(cm, class_names=["background", "no-dmg", "minor", "major", "destroyed"],
                      title="Damage segmentation - test confusion",
                      save_path=WORK / name / "test_confusion.png")
print("confusion saved")
mark("damage:eval-done")

In [ ]:
# ==== change detection (smoke) ====

In [ ]:
import logging
import pandas as pd
import torch as _torch
import torch.nn as _nn
import torch.nn.functional as _F
import segmentation_models_pytorch as _smp
from torch.utils.data import DataLoader
from rescuevision.data.datasets import XBDChangeDetectionDataset
from rescuevision.models.losses import BCEDiceLoss
from rescuevision.training.train_change_detection import train_change_detection
from rescuevision.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
OUTPUT = WORK
name = "change_detection_siamese_unet"
METRIC = "iou"


class SiameseUNet(_nn.Module):
    # Siamese U-Net: shared encoder, fused pre/post features, U-Net decoder.
    # Fixed decoder: skips deepest -> shallowest, concat fusion, consistent channels.

    def __init__(self, encoder_name="resnet34", encoder_weights="imagenet",
                 fusion_mode="concat_absdiff", classes=1):
        super().__init__()
        self.fusion_mode = fusion_mode
        self.classes = classes

        base_unet = _smp.Unet(encoder_name=encoder_name, encoder_weights=encoder_weights,
                              in_channels=3, classes=classes)
        self.encoder = base_unet.encoder
        encoder_channels = self._get_encoder_channels()
        bottleneck_ch = encoder_channels[-1]

        if fusion_mode == "concat":
            fusion_mult = 2
        elif fusion_mode == "absdiff":
            fusion_mult = 1
        elif fusion_mode == "concat_absdiff":
            fusion_mult = 3
        else:
            raise ValueError(f"Unknown fusion mode: {fusion_mode}")

        skip_channels = list(reversed(encoder_channels[:-1]))   # deepest -> shallowest
        decoder_channels = [256, 128, 64, 32]

        self.skip_convs = _nn.ModuleList()
        for enc_ch in skip_channels:
            self.skip_convs.append(_nn.Sequential(
                _nn.Conv2d(enc_ch * fusion_mult, enc_ch, 1, bias=False),
                _nn.BatchNorm2d(enc_ch),
                _nn.ReLU(inplace=True),
            ))

        self.decoder = _nn.ModuleList()
        in_ch = bottleneck_ch * fusion_mult + skip_channels[0]
        for i, dec_ch in enumerate(decoder_channels):
            self.decoder.append(_nn.Sequential(
                _nn.Conv2d(in_ch, dec_ch, 3, padding=1, bias=False),
                _nn.BatchNorm2d(dec_ch),
                _nn.ReLU(inplace=True),
                _nn.Conv2d(dec_ch, dec_ch, 3, padding=1, bias=False),
                _nn.BatchNorm2d(dec_ch),
                _nn.ReLU(inplace=True),
            ))
            if i + 1 < len(decoder_channels):
                in_ch = dec_ch + skip_channels[i + 1]

        self.segmentation_head = _nn.Conv2d(decoder_channels[-1], classes, 1)

    def _get_encoder_channels(self):
        channels = []
        dummy = _torch.randn(1, 3, 64, 64)
        with _torch.no_grad():
            features = self.encoder(dummy)
        return [f.shape[1] for f in features]

    def _fuse(self, a, b):
        if self.fusion_mode == "concat":
            return _torch.cat([a, b], dim=1)
        if self.fusion_mode == "absdiff":
            return _torch.abs(b - a)
        return _torch.cat([a, b, _torch.abs(b - a)], dim=1)  # concat_absdiff

    def forward(self, pre_image, post_image):
        features_pre = self.encoder(pre_image)
        features_post = self.encoder(post_image)
        x = self._fuse(features_pre[-1], features_post[-1])
        for i, (dec_block, skip_conv) in enumerate(zip(self.decoder, self.skip_convs)):
            x = _F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            skip = skip_conv(self._fuse(features_pre[-(i + 2)], features_post[-(i + 2)]))
            if x.shape[2:] != skip.shape[2:]:
                x = _F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
            x = dec_block(_torch.cat([x, skip], dim=1))
        if x.shape[2:] != pre_image.shape[2:]:
            x = _F.interpolate(x, size=pre_image.shape[2:], mode="bilinear", align_corners=False)
        return self.segmentation_head(x)


SEED = 42
AUG = {"horizontal_flip": 0.5, "vertical_flip": 0.5, "rotate90": 0.5, "brightness_contrast": 0.3, "normalize": True}
BS = 4
EPOCHS = 1
PATIENCE = 1

config = {
    "seed": SEED, "device": DEVICE.type,
    "training": {"epochs": EPOCHS, "lr": 1e-4, "weight_decay": 1e-5, "scheduler": "cosine",
                  "mixed_precision": True, "early_stopping_patience": PATIENCE,
                  "save_best_metric": METRIC},
}

set_seed(SEED)
split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"
train_ds = XBDChangeDetectionDataset(PROCESSED, split_csv, split="train",
                                     transform=joint_seg_transforms(AUG, "train", 512))
val_ds = XBDChangeDetectionDataset(PROCESSED, split_csv, split="val",
                                   transform=joint_seg_transforms(AUG, "val", 512))
test_ds = XBDChangeDetectionDataset(PROCESSED, split_csv, split="test",
                                    transform=joint_seg_transforms(AUG, "val", 512))
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
mark("change:data", extra=f"train={len(train_ds)}")

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
test_loader = DataLoader(test_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False)

model = SiameseUNet(encoder_name="resnet34", encoder_weights="imagenet",
                    fusion_mode="concat_absdiff", classes=1).to(DEVICE)
loss_fn = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)


try:
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        with torch.amp.autocast(DEVICE.type, enabled=True):
            _lg = model(_b["pre_image"].to(DEVICE), _b["post_image"].to(DEVICE))
            _loss = loss_fn(_lg.squeeze(1), _b["mask"].to(DEVICE).float())
        _loss.backward()
        model.zero_grad(set_to_none=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        mark("probe:ok", extra=f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    else:
        mark("probe:ok", extra="cpu")
except Exception:
    mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
    raise
mark("change:model-ready")
mark("change:train-start", extra=f"train={len(train_ds)}")
best = train_change_detection(model, train_loader, val_loader, loss_fn, config,
                              output_dir=str(Path("/tmp/run") / name))
print("best:", best)
mark("change:train-done", extra=str(best))

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/change_detection_siamese_unet", WORK / "change_detection_siamese_unet", dirs_exist_ok=True)
print("results copied to", WORK / "change_detection_siamese_unet")
mark("copyout:change_detection_siamese_unet")
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log_df = pd.read_csv(next((WORK / "change_detection_siamese_unet" / "logs").glob("*.csv")))
metric = "iou"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
if "val_loss" in log_df:
    axes[0].plot(log_df["epoch"], log_df["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(log_df["epoch"], log_df[metric], color="tab:green")
axes[1].set_title(f"val {metric} (best={log_df[metric].max():.4f} @ epoch {int(log_df.loc[log_df[metric].idxmax(), 'epoch'])})")
axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK / "change_detection_siamese_unet" / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import json

ckpt = torch.load(WORK / name / "checkpoints" / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
print(f"best checkpoint: epoch {ckpt['epoch']}, {METRIC}={ckpt.get(f'best_{METRIC}'):.4f}")

summary = {
    "run": name,
    "best_epoch": int(ckpt["epoch"]),
    f"best_val_{METRIC}": float(ckpt.get(f"best_{METRIC}")),
    "history_best": float(pd.read_csv(next((WORK / name / "logs").glob("*.csv")))[METRIC].max()),
}
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

import numpy as np, matplotlib.pyplot as plt
from rescuevision.evaluation.metrics_segmentation import compute_segmentation_metrics_batch

@torch.no_grad()
def eval_change(model, loader, device):
    model.eval()
    agg, n = {}, 0
    for batch in loader:
        pre = batch["pre_image"].to(device)
        post = batch["post_image"].to(device)
        masks = batch["mask"].to(device)
        logits = model(pre, post)
        metrics = compute_segmentation_metrics_batch(logits, masks, num_classes=1, is_binary=True)
        for k, v in metrics.items():
            agg[k] = agg.get(k, 0.0) + v * pre.shape[0]
        n += pre.shape[0]
    return {k: v / max(n, 1) for k, v in agg.items()}

val_metrics = eval_change(model, val_loader, DEVICE)
test_metrics = eval_change(model, test_loader, DEVICE)
print("VAL :", {k: round(v, 4) for k, v in val_metrics.items()})
print("TEST:", {k: round(v, 4) for k, v in test_metrics.items()})
summary.update({"val": {k: round(v, 4) for k, v in val_metrics.items()},
                "test": {k: round(v, 4) for k, v in test_metrics.items()}})
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

batch = next(iter(test_loader))
with torch.no_grad():
    pred = (torch.sigmoid(model(batch["pre_image"].to(DEVICE), batch["post_image"].to(DEVICE))) > 0.5).cpu().numpy()
def denorm(t):
    img = t.numpy().transpose(1, 2, 0)
    return (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
fig, axes = plt.subplots(4, 4, figsize=(18, 18))
for i in range(min(4, batch["pre_image"].shape[0])):
    axes[0, i].imshow(denorm(batch["pre_image"][i])); axes[0, i].set_title(batch["tile_id"][i][:24])
    axes[1, i].imshow(denorm(batch["post_image"][i])); axes[1, i].set_title("post")
    axes[2, i].imshow(batch["mask"][i], cmap="gray"); axes[2, i].set_title("GT change")
    axes[3, i].imshow(pred[i, 0], cmap="gray"); axes[3, i].set_title("pred change")
    for ax in axes[:, i]: ax.axis("off")
plt.tight_layout(); plt.savefig(WORK / name / "test_predictions.png", dpi=120, bbox_inches="tight")
plt.show()
mark("change:eval-done")

In [ ]:
# ==== dinov2 classifier (smoke, 2 epochs to exercise unfreeze stage) ====

In [ ]:
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD
from rescuevision.data import datasets as _D
from PIL import Image as _PIL
from pathlib import Path as _PPath

def _load_arr(path):
    """Read .png or .npy array transparently."""
    path = _PPath(path)
    if path.suffix == ".npy":
        return np.load(path)
    return np.array(_PIL.open(path))

def _find_tile(directory, tile_id):
    for ext in (".png", ".npy"):
        p = _PPath(directory) / f"{tile_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{tile_id} in {directory}")

def joint_seg_transforms(cfg, phase="train", image_size=512):
    """Joint pipeline for a stacked pre+post pair (H, W, 6): one geometric draw for both."""
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(image_size, image_size)]
    if phase == "train":
        if cfg.get("horizontal_flip", 0) > 0:
            tf.append(A.HorizontalFlip(p=cfg["horizontal_flip"]))
        if cfg.get("vertical_flip", 0) > 0:
            tf.append(A.VerticalFlip(p=cfg["vertical_flip"]))
        if cfg.get("rotate90", 0) > 0:
            tf.append(A.RandomRotate90(p=cfg["rotate90"]))
        if cfg.get("brightness_contrast", 0) > 0:
            tf.append(A.RandomBrightnessContrast(p=cfg["brightness_contrast"]))
    if cfg.get("normalize", True):
        tf.append(A.Normalize(mean=mean, std=std))
    tf.append(ToTensorV2())
    return A.Compose(tf, is_check_shapes=False)

def joint_clf_pair_transforms(cfg, phase="train", image_size=224):
    """Joint pipeline for stacked pre+post crops (H, W, 6)."""
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(image_size, image_size)]
    if phase == "train":
        if cfg.get("horizontal_flip", 0) > 0:
            tf.append(A.HorizontalFlip(p=cfg["horizontal_flip"]))
        if cfg.get("vertical_flip", 0) > 0:
            tf.append(A.VerticalFlip(p=cfg["vertical_flip"]))
        if cfg.get("rotate", 0) > 0:
            tf.append(A.Rotate(limit=cfg["rotate"], p=0.5))
        if cfg.get("brightness_contrast", 0) > 0:
            tf.append(A.RandomBrightnessContrast(p=cfg["brightness_contrast"]))
    if cfg.get("normalize", True):
        tf.append(A.Normalize(mean=mean, std=std))
    tf.append(ToTensorV2())
    return A.Compose(tf, is_check_shapes=False)

def _nocheck(tf):
    # disable albumentations' shape pre-check: newer builds (preinstalled on the
    # Kaggle GPU image) reject our verified HxW-consistent image/mask pairs
    try:
        return A.Compose(tf.transforms, is_check_shapes=False)
    except Exception:
        return tf

def _joint_dmg_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    pre_image = _load_arr(_find_tile(self.pre_dir, tile_id))
    post_image = _load_arr(_find_tile(self.post_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    pair = np.concatenate([pre_image, post_image], axis=2)
    if self.transform:
        augmented = self.transform(image=pair, mask=mask)
        image, mask = augmented["image"], augmented["mask"]
    else:
        image = torch.from_numpy(pair.transpose(2, 0, 1))
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

def _joint_change_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    pre_image = _load_arr(_find_tile(self.pre_dir, tile_id))
    post_image = _load_arr(_find_tile(self.post_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    pair = np.concatenate([pre_image, post_image], axis=2)
    if self.transform:
        augmented = self.transform(image=pair, mask=mask)
        pair_t, mask = augmented["image"], augmented["mask"]
    else:
        pair_t = torch.from_numpy(pair.transpose(2, 0, 1))
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"pre_image": pair_t[:3], "post_image": pair_t[3:], "mask": mask, "tile_id": tile_id}

_D.XBDDamageSegmentationDataset.__getitem__ = _joint_dmg_seg_getitem
_D.XBDChangeDetectionDataset.__getitem__ = _joint_change_getitem

def _joint_crop_getitem(self, idx):
    row = self.crops_df.iloc[idx]
    pre_crop = _load_arr(row["pre_crop_path"])
    post_crop = _load_arr(row["post_crop_path"])
    label = self.label_map[row["label_id"]]
    pair = np.concatenate([pre_crop, post_crop], axis=2)
    if self.transform:
        pair_t = self.transform(image=pair)["image"]
    else:
        pair_t = torch.from_numpy(pair.transpose(2, 0, 1))
    return {"pre_crop": pair_t[:3], "post_crop": pair_t[3:], "label": label,
            "crop_id": row["crop_id"], "tile_id": row["tile_id"]}

_D.XBDDamageCropDataset.__getitem__ = _joint_crop_getitem

def _tile_dataset_init(self, processed_dir, split_csv, split="train", mask_type="building_binary",
                       transform=None, image_size=512, debug_limit=None):
    import pandas as pd
    self.processed_dir = _PPath(processed_dir)
    self.images_dir = self.processed_dir / "images" / "pre"
    self.pre_dir = self.processed_dir / "images" / "pre"
    self.post_dir = self.processed_dir / "images" / "post"
    if type(self).__name__ == "XBDDamageSegmentationDataset":
        self.masks_dir = self.processed_dir / "masks" / "damage_multiclass"
    elif type(self).__name__ == "XBDChangeDetectionDataset":
        self.masks_dir = self.processed_dir / "masks" / "change_binary"
    else:
        self.masks_dir = self.processed_dir / "masks" / mask_type
    self.transform = transform
    self.image_size = image_size
    splits_df = pd.read_csv(split_csv)
    tile_ids = splits_df[splits_df["split"] == split]["tile_id"].tolist()
    if debug_limit:
        tile_ids = tile_ids[:debug_limit]
    self.tile_ids = [t for t in tile_ids
                     if _find_tile(self.pre_dir, t).exists()
                     and _find_tile(self.post_dir, t).exists()
                     and _find_tile(self.masks_dir, t).exists()]
    print(f"{type(self).__name__}: split={split}, {len(self.tile_ids)} tiles")

def _xbd_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    image = _load_arr(_find_tile(self.pre_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    if self.transform:
        aug = self.transform(image=image, mask=mask)
        image, mask = aug["image"], aug["mask"]
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

_D.XBDSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDDamageSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDChangeDetectionDataset.__init__ = _tile_dataset_init
_D.XBDSegmentationDataset.__getitem__ = _xbd_seg_getitem
mark("patch:done")
print("png-aware dataset init patch applied")
print("joint pre/post transform patch applied")

In [ ]:
import logging
import pandas as pd
from torch.utils.data import DataLoader
from rescuevision.data.datasets import XBDDamageCropDataset
from rescuevision.models.dinov2_classifier import DINOv2Classifier
from rescuevision.models.losses import FocalLoss, compute_class_weights
from rescuevision.training.train_damage_classifier import train_damage_classifier
from rescuevision.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
OUTPUT = WORK
name = "damage_classifier_dinov2"
METRIC = "macro_f1"

SEED = 42
AUG = {"horizontal_flip": 0.5, "vertical_flip": 0.5, "rotate": 15, "brightness_contrast": 0.3, "normalize": True}
BS = 8
MAX_CROPS = 1000000000     # quota-friendly cap, deterministic subsample
EPOCHS = 2           # stage 1: frozen head | stage 2: last 4 blocks unfrozen
UNFREEZE_AFTER = 1
PATIENCE = 2

config = {
    "seed": SEED, "device": DEVICE.type,
    "model": {"unfreeze_after_epoch": UNFREEZE_AFTER, "unfreeze_blocks": 4},
    "training": {"epochs": EPOCHS, "lr": 1e-3, "stage2_lr": 1e-5, "weight_decay": 1e-4,
                  "scheduler": "cosine", "mixed_precision": True,
                  "early_stopping_patience": PATIENCE, "save_best_metric": METRIC},
}

set_seed(SEED)
# rewrite relative crop paths from the processed tree to absolute
crops = pd.read_csv(PROCESSED / "metadata" / "buildings.csv")
splits = pd.read_csv(PROCESSED / "metadata" / "splits_event_aware.csv")
crops["pre_crop_path"] = str(PROCESSED) + "/" + crops["pre_crop_path"]
crops["post_crop_path"] = str(PROCESSED) + "/" + crops["post_crop_path"]
crops_csv = OUTPUT / "building_crops_mounted.csv"
crops.to_csv(crops_csv, index=False)
print(f"{len(crops)} classified crops total")

split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"
train_ds = XBDDamageCropDataset(crops_csv, split_csv, split="train",
                                transform=joint_clf_pair_transforms(AUG, "train", 224))
val_ds = XBDDamageCropDataset(crops_csv, split_csv, split="val",
                              transform=joint_clf_pair_transforms(AUG, "val", 224))
test_ds = XBDDamageCropDataset(crops_csv, split_csv, split="test",
                               transform=joint_clf_pair_transforms(AUG, "val", 224))

# deterministic cap on train crops only (keep val/test intact)
if len(train_ds) > MAX_CROPS:
    rng = np.random.default_rng(SEED)
    keep = rng.permutation(len(train_ds))[:MAX_CROPS]
    train_ds.crops_df = train_ds.crops_df.iloc[sorted(keep)].reset_index(drop=True)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
mark("dino:data", extra=f"train={len(train_ds)}")

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
test_loader = DataLoader(test_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False)

label_names = ["no-damage", "minor-damage", "major-damage", "destroyed"]
train_labels = train_ds.crops_df["label_id"].map({1: 0, 2: 1, 3: 2, 4: 3}).tolist()
class_weights = compute_class_weights(train_labels, num_classes=4).to(DEVICE)
print("class weights:", [round(w, 3) for w in class_weights.tolist()])

model = DINOv2Classifier(backbone="facebook/dinov2-base", num_classes=4,
                         freeze_backbone=True, mlp_hidden_dim=512, dropout=0.3).to(DEVICE)
loss_fn = FocalLoss(gamma=2.0, alpha=class_weights)


try:
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        with torch.amp.autocast(DEVICE.type, enabled=True):
            _lg = model(_b["pre_crop"].to(DEVICE), _b["post_crop"].to(DEVICE))
            _loss = loss_fn(_lg, _b["label"].to(DEVICE))
        _loss.backward()
        model.zero_grad(set_to_none=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        mark("probe:ok", extra=f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    else:
        mark("probe:ok", extra="cpu")
except Exception:
    mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
    raise
mark("dino:model-ready")
mark("dino:train-start", extra=f"train={len(train_ds)}")
best = train_damage_classifier(model, train_loader, val_loader, loss_fn, config,
                               output_dir=str(Path("/tmp/run") / name), class_names=label_names)
print("best:", best)
mark("dino:train-done", extra=str(best))

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/damage_classifier_dinov2", WORK / "damage_classifier_dinov2", dirs_exist_ok=True)
print("results copied to", WORK / "damage_classifier_dinov2")
mark("copyout:damage_classifier_dinov2")
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log_df = pd.read_csv(next((WORK / "damage_classifier_dinov2" / "logs").glob("*.csv")))
metric = "macro_f1"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
if "val_loss" in log_df:
    axes[0].plot(log_df["epoch"], log_df["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(log_df["epoch"], log_df[metric], color="tab:green")
axes[1].set_title(f"val {metric} (best={log_df[metric].max():.4f} @ epoch {int(log_df.loc[log_df[metric].idxmax(), 'epoch'])})")
axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK / "damage_classifier_dinov2" / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import json

ckpt = torch.load(WORK / name / "checkpoints" / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
print(f"best checkpoint: epoch {ckpt['epoch']}, {METRIC}={ckpt.get(f'best_{METRIC}'):.4f}")

summary = {
    "run": name,
    "best_epoch": int(ckpt["epoch"]),
    f"best_val_{METRIC}": float(ckpt.get(f"best_{METRIC}")),
    "history_best": float(pd.read_csv(next((WORK / name / "logs").glob("*.csv")))[METRIC].max()),
}
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

import numpy as np
from rescuevision.training.train_damage_classifier import _evaluate_classifier
val_metrics = _evaluate_classifier(model, val_loader, DEVICE, loss_fn, True)
test_metrics = _evaluate_classifier(model, test_loader, DEVICE, loss_fn, True)
print("VAL :", {k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)})
print("TEST:", {k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)})

y_true, y_pred = [], []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        with torch.amp.autocast(DEVICE.type, enabled=True):
            logits = model(batch["pre_crop"].to(DEVICE), batch["post_crop"].to(DEVICE))
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(batch["label"].numpy())
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
import seaborn as sns, matplotlib.pyplot as plt
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.title("DINOv2 classifier - test confusion"); plt.tight_layout()
plt.savefig(WORK / name / "test_confusion.png", dpi=120, bbox_inches="tight")
plt.show()
mark("dino:eval-done")

In [ ]:
# ==== floodnet segformer (smoke) ====

In [ ]:
# ---- synthetic mini FloodNet (grayscale masks, like the real aletbm set) ----
import numpy as np
from PIL import Image

SRC = Path("/tmp/floodnet_src")
for grp in ["Flooded", "Non-Flooded"]:
    (SRC / "Train/Labeled" / grp / "image").mkdir(parents=True, exist_ok=True)
    (SRC / "Train/Labeled" / grp / "mask").mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(1)
for i in range(8):
    grp = "Flooded" if i % 2 == 0 else "Non-Flooded"
    stem = f"synth_{i:04d}"
    Image.fromarray(rng.integers(0, 255, (1024, 1024, 3), dtype=np.uint8)).save(
        SRC / "Train/Labeled" / grp / "image" / f"{stem}.jpg", quality=85)
    m = np.zeros((1024, 1024), dtype=np.uint8)
    for c in range(10):   # class blocks + noise -> >=2 classes per 512 tile
        y, x = rng.integers(0, 512, 2)
        m[y:y+200, x:x+200] = rng.integers(0, 10)
    noise = rng.random((1024, 1024)) < 0.02
    m[noise] = rng.integers(0, 10, size=int(noise.sum()))
    Image.fromarray(m).save(SRC / "Train/Labeled" / grp / "mask" / f"{stem}_lab.png")
print("synthetic floodnet: 8 images")
mark("synthetic:floodnet-done")

In [ ]:
import numpy as np, pandas as pd, random
from PIL import Image
from pathlib import Path

ROOT = None
for _c in Path("/kaggle/input").rglob("class_mapping.csv"):
    ROOT = _c.parent
    break
if ROOT is None and Path("/tmp/floodnet_src").exists():
    ROOT = Path("/tmp/floodnet_src")          # synthetic smoke path
assert ROOT is not None, "FloodNet dataset not found under /kaggle/input"
print("FloodNet root:", ROOT)
mark("floodnet:root", extra=str(ROOT))
TILES = Path("/tmp/floodnet_tiles")
(TILES / "train").mkdir(parents=True, exist_ok=True)
(TILES / "val").mkdir(parents=True, exist_ok=True)

from rescuevision.constants import FLOODNET_CLASSES, NUM_FLOODNET_CLASSES

import re
def norm_name(s):
    return re.sub(r"[^a-z]", "", str(s).lower())

NAME_TO_ID = {norm_name(v): k for k, v in FLOODNET_CLASSES.items()}

OFFICIAL_PALETTE = {  # official FloodNet colors, fallback if the CSV can't be parsed
    (0, 0, 0): 0, (72, 0, 255): 1, (255, 0, 0): 2, (0, 255, 255): 3, (255, 255, 0): 4,
    (0, 255, 0): 5, (0, 0, 255): 6, (255, 0, 255): 7, (0, 128, 128): 8, (128, 128, 128): 9,
}

color_to_id = {}
try:
    cm = pd.read_csv(ROOT / "class_mapping.csv")
    print("class_mapping.csv:\n", cm.to_string())
    cols = {c.lower(): c for c in cm.columns}
    name_col = next((cols[c] for c in cols if "name" in c or "class" in c), None)
    has_rgb = all(any(k in cols for k in (key,)) for key in ("r", "g", "b"))
    hex_col = next((cols[c] for c in cols if "hex" in c or "color" in c), None)
    for _, row in cm.iterrows():
        cid = NAME_TO_ID.get(norm_name(row[name_col])) if name_col else None
        if cid is None:
            continue
        if has_rgb:
            color_to_id[(int(row[cols["r"]]), int(row[cols["g"]]), int(row[cols["b"]]))] = cid
        elif hex_col and str(row[hex_col]).startswith("#"):
            h = str(row[hex_col]).lstrip("#")
            color_to_id[tuple(int(h[i:i+2], 16) for i in (0, 2, 4))] = cid
except Exception as e:
    print("could not parse class_mapping.csv:", e)
if len(color_to_id) < NUM_FLOODNET_CLASSES and (ROOT / "class_mapping.csv").exists() is False:
    print("no class_mapping.csv - masks assumed grayscale ids")
elif len(color_to_id) < NUM_FLOODNET_CLASSES:
    print("falling back to official FloodNet palette")
    color_to_id = OFFICIAL_PALETTE

def mask_to_ids(mask_rgb):
    # (H, W, 3) uint8 -> (H, W) class ids, nearest palette color
    m = mask_rgb.astype(np.int16)
    ids = np.zeros(m.shape[:2], dtype=np.uint8)
    best = np.full(m.shape[:2], 3 * 255, dtype=np.int16)
    for color, cid in color_to_id.items():
        d = (np.abs(m[:, :, 0] - color[0]) + np.abs(m[:, :, 1] - color[1])
             + np.abs(m[:, :, 2] - color[2])).astype(np.int16)
        better = d < best
        ids[better] = cid
        np.minimum(best, d, out=best)
    return ids

sample_mask = next((ROOT / "Train/Labeled/Flooded/mask").glob("*.png"))
sm = Image.open(sample_mask)
print("sample mask mode:", sm.mode, sm.size)
grayscale_mode = sm.mode in ("L", "P")
print("grayscale_mode:", grayscale_mode)
mark("floodnet:format", extra=f"grayscale={grayscale_mode}")

In [ ]:
import random, os
from tqdm.auto import tqdm

TILE = 512
SEED = 42
VAL_FRAC = 0.15
MAX_TRAIN_TILES = 1000000000    # quota-friendly caps (flood tiles kept with priority)
MAX_VAL_TILES = 1000000000
MAX_TMP_GB = 40       # /tmp safety guard

img_dirs = [ROOT / "Train/Labeled/Flooded", ROOT / "Train/Labeled/Non-Flooded"]
samples = []
for d in img_dirs:
    for img_path in sorted((d / "image").glob("*.jpg")):
        mask_path = d / "mask" / (img_path.stem + "_lab.png")   # official naming: {id}_lab.png
        if not mask_path.exists():
            mask_path = d / "mask" / (img_path.stem + ".png")
        if mask_path.exists():
            samples.append((img_path, mask_path))
print(f"{len(samples)} labeled images")
mark("floodnet:samples", extra=str(len(samples)))

rng = random.Random(SEED)
idx = rng.sample(range(len(samples)), max(1, int(len(samples) * VAL_FRAC)))
val_set = set(idx)

def tree_bytes(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())

records = []   # (img_npy, ids_npy, split, is_flood, bincount_row)
n_train = n_val = 0
for i, (img_path, mask_path) in enumerate(tqdm(samples, desc="tiling")):
    split = "val" if i in val_set else "train"
    img = np.array(Image.open(img_path).convert("RGB"))
    msk_img = Image.open(mask_path)
    if grayscale_mode:
        ids = np.array(msk_img.convert("L"), dtype=np.uint8)
    else:
        ids = mask_to_ids(np.array(msk_img.convert("RGB")))
    ids = np.clip(ids, 0, NUM_FLOODNET_CLASSES - 1)  # guard stray values
    H, W = img.shape[:2]
    for y in range(0, H - TILE + 1, TILE):
        for x in range(0, W - TILE + 1, TILE):
            tile_ids = ids[y:y+TILE, x:x+TILE]
            if len(np.unique(tile_ids)) < 2:
                continue  # skip fully-uniform tiles (usually all background)
            stem = f"{img_path.stem}_{y//TILE}_{x//TILE}"
            img_npy = TILES / split / f"{stem}_img.npy"
            ids_npy = TILES / split / f"{stem}_ids.npy"
            np.save(img_npy, img[y:y+TILE, x:x+TILE])
            np.save(ids_npy, tile_ids)
            cnt = np.bincount(tile_ids.flatten(), minlength=NUM_FLOODNET_CLASSES)
            records.append((img_npy, ids_npy, split, bool(cnt[1] + cnt[3] + cnt[5] > 0), cnt))
            if split == "train": n_train += 1
            else: n_val += 1
    if (i + 1) % 200 == 0 and tree_bytes(TILES) > MAX_TMP_GB * 1024**3:
        print(f"/tmp size guard hit at image {i+1}, stopping tiling early")
        mark("floodnet:tiling-guard", extra=str(i))
        break

print(f"tiles before cap: train={n_train} val={n_val}")

# cap train tiles: keep ALL flood tiles, subsample the rest deterministically
rng2 = np.random.default_rng(SEED)
train_recs = [r for r in records if r[2] == "train"]
if len(train_recs) > MAX_TRAIN_TILES:
    flood = [r for r in train_recs if r[3]]
    rest = [r for r in train_recs if not r[3]]
    budget = max(0, MAX_TRAIN_TILES - len(flood))
    keep = set(map(id, flood)) | set(map(id, rng2.choice(rest, size=min(budget, len(rest)), replace=False) if len(rest) else []))
    for r in train_recs:
        if id(r) not in keep:
            r[0].unlink(missing_ok=True); r[1].unlink(missing_ok=True)
    train_recs = [r for r in train_recs if id(r) in keep]
val_recs = [r for r in records if r[2] == "val"]
if len(val_recs) > MAX_VAL_TILES:
    keep = set(map(id, rng2.choice(val_recs, size=MAX_VAL_TILES, replace=False)))
    for r in val_recs:
        if id(r) not in keep:
            r[0].unlink(missing_ok=True); r[1].unlink(missing_ok=True)
    val_recs = [r for r in val_recs if id(r) in keep]

counts = np.sum([r[4] for r in train_recs], axis=0).astype(np.int64) if train_recs else np.zeros(NUM_FLOODNET_CLASSES, dtype=np.int64)
print(f"tiles after cap: train={len(train_recs)} val={len(val_recs)} | tree {tree_bytes(TILES)/1e9:.1f} GB")
print("class pixel counts:", counts.tolist())
assert counts.sum() > 0
mark("floodnet:tiles", extra=f"train={len(train_recs)} val={len(val_recs)}")

In [ ]:
import logging
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from rescuevision.models.losses import WeightedCEDiceLoss
from rescuevision.training.train_segmentation import train_segmentation, evaluate_segmentation
from rescuevision.utils.seed import set_seed
mark("floodnet:imports")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
OUTPUT = WORK
name = "floodnet_segformer_b2"
METRIC = "miou"

SEED = 42
AUG = {"horizontal_flip": 0.5, "vertical_flip": 0.5, "rotate90": 0.5, "brightness_contrast": 0.3, "normalize": True}
BS = 2
EPOCHS = 1
PATIENCE = 1

class FloodNetTileDataset(Dataset):
    def __init__(self, split_dir, transform=None):
        self.files = sorted(split_dir.glob("*_img.npy"))
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        base = str(self.files[i])[: -len("_img.npy")]   # strip the _img.npy suffix
        image = np.load(base + "_img.npy")
        mask = np.load(base + "_ids.npy")
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug["image"], aug["mask"]
        mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
        return {"image": image, "mask": mask, "tile_id": Path(base).name}

from rescuevision.data.transforms import get_segmentation_transforms

set_seed(SEED)
train_ds = FloodNetTileDataset(TILES / "train", _nocheck(get_segmentation_transforms(AUG, "train", 512)))
val_ds = FloodNetTileDataset(TILES / "val", _nocheck(get_segmentation_transforms(AUG, "val", 512)))
print(f"train={len(train_ds)} val={len(val_ds)}")
mark("floodnet:data", extra=f"train={len(train_ds)}")
assert len(train_ds) > 0 and len(val_ds) > 0

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False)

# recompute class counts from the extracted tiles (self-contained cell)
counts = np.zeros(NUM_FLOODNET_CLASSES, dtype=np.int64)
for _f in (TILES / "train").glob("*_ids.npy"):
    counts += np.bincount(np.load(_f).flatten(), minlength=NUM_FLOODNET_CLASSES)
print("class pixel counts:", counts.tolist())
assert counts.sum() > 0
freq = counts / counts.sum()
weights = 1.0 / np.sqrt(freq)
weights = weights / weights.sum() * NUM_FLOODNET_CLASSES
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
print("class weights:", [round(w, 3) for w in class_weights.tolist()])

# SMP SegFormer (mit_b2): the HF-transformers SegFormer hard-crashed CUDA on the
# Kaggle T4 image 4/4 times while SMP trained fine (building seg canary)
model = smp.Segformer(encoder_name="mit_b2", encoder_weights="imagenet",
                      in_channels=3, classes=NUM_FLOODNET_CLASSES).to(DEVICE)

loss_fn = WeightedCEDiceLoss(class_weights=class_weights, ce_weight=0.5, dice_weight=0.5, ignore_index=255)

config = {"seed": SEED, "device": DEVICE.type,
          "training": {"epochs": EPOCHS, "lr": 6e-5, "weight_decay": 1e-4, "scheduler": "cosine",
                       "mixed_precision": True, "early_stopping_patience": PATIENCE,
                       "save_best_metric": METRIC, "gradient_accumulation_steps": 1}}


# first-batch GPU probe: an OOM or crash here lands in the marks WITH a traceback
try:
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        _img = _b["image"].to(DEVICE)
        _m = _b["mask"].to(DEVICE)
        with torch.amp.autocast(DEVICE.type, enabled=True):
            _lg = model(_img)
            _loss = loss_fn(_lg.squeeze(1) if _lg.shape[1] == 1 else _lg,
                            _m.float() if _lg.shape[1] == 1 else _m)
        _loss.backward()
        model.zero_grad(set_to_none=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        mark("probe:ok", extra=f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    else:
        mark("probe:ok", extra="cpu")
except Exception:
    mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
    raise
mark("floodnet:model-ready")
mark("floodnet:train-start", extra=f"train={len(train_ds)}")
best = train_segmentation(model, train_loader, val_loader, loss_fn, config,
                          output_dir=str(Path("/tmp/run") / name), is_binary=False,
                          num_classes=NUM_FLOODNET_CLASSES, ignore_index=255)
print("best:", best)
mark("floodnet:train-done", extra=str(best))

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/floodnet_segformer_b2", WORK / "floodnet_segformer_b2", dirs_exist_ok=True)
print("results copied to", WORK / "floodnet_segformer_b2")
mark("copyout:floodnet_segformer_b2")
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log_df = pd.read_csv(next((WORK / "floodnet_segformer_b2" / "logs").glob("*.csv")))
metric = "miou"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
if "val_loss" in log_df:
    axes[0].plot(log_df["epoch"], log_df["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(log_df["epoch"], log_df[metric], color="tab:green")
axes[1].set_title(f"val {metric} (best={log_df[metric].max():.4f} @ epoch {int(log_df.loc[log_df[metric].idxmax(), 'epoch'])})")
axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK / "floodnet_segformer_b2" / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import json

ckpt = torch.load(WORK / name / "checkpoints" / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
print(f"best checkpoint: epoch {ckpt['epoch']}, {METRIC}={ckpt.get(f'best_{METRIC}'):.4f}")

summary = {
    "run": name,
    "best_epoch": int(ckpt["epoch"]),
    f"best_val_{METRIC}": float(ckpt.get(f"best_{METRIC}")),
    "history_best": float(pd.read_csv(next((WORK / name / "logs").glob("*.csv")))[METRIC].max()),
}
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

import numpy as np
val_metrics = evaluate_segmentation(model, val_loader, DEVICE, is_binary=False,
                                    num_classes=NUM_FLOODNET_CLASSES, ignore_index=255)
print("VAL :", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in val_metrics.items()})
summary.update({"val": {k: round(v, 4) for k, v in val_metrics.items() if not isinstance(v, list)}})
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

from rescuevision.constants import FLOODNET_CLASSES, FLOODNET_COLOR_MAP
import matplotlib.pyplot as plt
class_ious = val_metrics.get("class_ious")
if class_ious:
    for cid, iou in zip(range(NUM_FLOODNET_CLASSES), class_ious):
        print(f"  {FLOODNET_CLASSES[cid]:22s} IoU={iou:.4f}")

batch = next(iter(val_loader))
with torch.no_grad():
    pred = model(batch["image"].to(DEVICE)).argmax(1).cpu().numpy()
def colorize(m):
    rgb = np.zeros((*m.shape, 3), dtype=np.uint8)
    for c, col in FLOODNET_COLOR_MAP.items():
        rgb[m == c] = col
    return rgb
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i in range(min(4, batch["image"].shape[0])):
    img = batch["image"][i].numpy().transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    axes[0, i].imshow(img); axes[0, i].set_title(batch["tile_id"][i][:24])
    axes[1, i].imshow(colorize(batch["mask"][i].numpy())); axes[1, i].set_title("GT")
    axes[2, i].imshow(colorize(pred[i])); axes[2, i].set_title("pred")
    for ax in axes[:, i]: ax.axis("off")
plt.tight_layout(); plt.savefig(WORK / name / "val_predictions.png", dpi=120, bbox_inches="tight")
plt.show()
mark("floodnet:eval-done")

In [ ]:
import json
ok = all(Path(WORK / n / "summary.json").exists() for n in
         ["building_seg_unet_rn34", "damage_seg_unet_effb3_6ch", "change_detection_siamese_unet",
          "damage_classifier_dinov2", "floodnet_segformer_b2"])
mark("smoke:ALL-DONE", ok=ok)
assert ok, "one or more smoke pipelines did not finish"
print("SMOKE PASSED: all 5 pipelines trained + evaluated on CPU")